In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split

import pickle

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/preprocessed_data.csv")

df.head()

,title,text,label,content,tokens,stemmed,lemmatized,clean_text
0,‘A target on Roe v. Wade ’: Oklahoma bill maki...,UPDATE: Gov. Fallin vetoed the bill on Friday....,REAL,a target on roe v wade oklahoma bill making i...,"['target', 'roe', 'v', 'wade', 'oklahoma', 'bi...","['target', 'roe', 'v', 'wade', 'oklahoma', 'bi...","['target', 'roe', 'v', 'wade', 'oklahoma', 'bi...",target roe v wade oklahoma bill making felony ...
1,Study: women had to drive 4 times farther afte...,Ever since Texas laws closed about half of the...,REAL,study women had to drive times farther after t...,"['study', 'women', 'drive', 'times', 'farther'...","['studi', 'women', 'drive', 'time', 'farther',...","['study', 'woman', 'drive', 'time', 'farther',...",study woman drive time farther texas law close...
2,"Trump, Clinton clash in dueling DC speeches","Donald Trump and Hillary Clinton, now at the s...",REAL,trump clinton clash in dueling dc speeches don...,"['trump', 'clinton', 'clash', 'dueling', 'dc',...","['trump', 'clinton', 'clash', 'duel', 'dc', 's...","['trump', 'clinton', 'clash', 'dueling', 'dc',...",trump clinton clash dueling dc speech donald t...
3,Grand jury in Texas indicts activists behind P...,A Houston grand jury investigating criminal al...,REAL,grand jury in texas indicts activists behind p...,"['grand', 'jury', 'texas', 'indicts', 'activis...","['grand', 'juri', 'texa', 'indict', 'activist'...","['grand', 'jury', 'texas', 'indicts', 'activis...",grand jury texas indicts activist behind plann...
4,"As Reproductive Rights Hang In The Balance, De...",WASHINGTON -- Forty-three years after the Supr...,REAL,as reproductive rights hang in the balance deb...,"['reproductive', 'rights', 'hang', 'balance', ...","['reproduct', 'right', 'hang', 'balanc', 'deba...","['reproductive', 'right', 'hang', 'balance', '...",reproductive right hang balance debate moderat...


In [3]:
df = df[["clean_text","label"]]

df.head()

,clean_text,label
0,target roe v wade oklahoma bill making felony ...,REAL
1,study woman drive time farther texas law close...,REAL
2,trump clinton clash dueling dc speech donald t...,REAL
3,grand jury texas indicts activist behind plann...,REAL
4,reproductive right hang balance debate moderat...,REAL


In [4]:
df.isnull().sum()

clean_text    0
label         0
dtype: int64

In [5]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["label"])

In [6]:
print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

{'FAKE': np.int64(0), 'REAL': np.int64(1)}


In [7]:
X = df["clean_text"]

y = df["label"]

In [8]:
count_vectorizer = CountVectorizer()

X_count = count_vectorizer.fit_transform(X)

In [9]:
print(X_count.shape)

(4593, 61974)


In [10]:
print(len(count_vectorizer.vocabulary_))

61974


In [11]:
feature_names = count_vectorizer.get_feature_names_out()

print(feature_names[:20])

['aa' 'aaa' 'aab' 'aabenraa' 'aacs' 'aadhar' 'aadmi' 'aae' 'aahing' 'aaj'
 'aakar' 'aakhri' 'aali' 'aaliya' 'aaliyas' 'aam' 'aand' 'aap' 'aapke'
 'aapko']


In [12]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

In [13]:
X_tfidf = tfidf.fit_transform(X)

In [14]:
print(X_tfidf.shape)

(4593, 5000)


In [15]:
print(tfidf.get_feature_names_out()[:20])

['abandon' 'abandoned' 'abc' 'abedin' 'ability' 'able' 'aboard' 'abortion'
 'abroad' 'absence' 'absent' 'absentee' 'absolute' 'absolutely' 'absurd'
 'abuse' 'academic' 'academy' 'accept' 'acceptable']


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [17]:
print("Training Features :", X_train.shape)

print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)

print("Testing Labels :", y_test.shape)

Training Features : (3674, 5000)
Testing Features : (919, 5000)
Training Labels : (3674,)
Testing Labels : (919,)


In [18]:
with open("../models/tfidf_vectorizer.pkl","wb") as file:
    pickle.dump(tfidf,file)

In [19]:
with open("../models/label_encoder.pkl","wb") as file:
    pickle.dump(encoder,file)

In [20]:
import os

print(os.listdir("../models"))

['label_encoder.pkl', 'tfidf_vectorizer.pkl']


In [21]:
df.to_csv(
    "../data/processed/final_dataset.csv",
    index=False
)

In [22]:
with open("../models/tfidf_vectorizer.pkl","rb") as file:
    loaded_vectorizer = pickle.load(file)

print("Vectorizer Loaded Successfully")

Vectorizer Loaded Successfully


In [23]:
sample = ["india won world cup"]

vector = loaded_vectorizer.transform(sample)

print(vector.shape)

(1, 5000)


In [24]:
print("="*60)

print("Feature Engineering Completed Successfully")

print("="*60)

print("Total Samples :", len(df))

print("Training Samples :", X_train.shape[0])

print("Testing Samples :", X_test.shape[0])

print("Vocabulary Size :", len(tfidf.get_feature_names_out()))

Feature Engineering Completed Successfully
Total Samples : 4593
Training Samples : 3674
Testing Samples : 919
Vocabulary Size : 5000
